In [59]:
import pandas as pd

df_match = pd.read_json("/Users/muaazshanji/projects/statsbomb-data/data/events/15946.json")
df_match["type_name"] = df_match["type"].apply(lambda x: x["name"] if pd.notna(x) else None)
df_match["player_name"] = df_match["player"].apply(lambda x: x["name"] if pd.notna(x) else None)
df_match["team_name"] = df_match["team"].apply(lambda x: x["name"] if pd.notna(x) else None)

In [60]:
print(df_match.groupby("team_name")["type_name"].count())

team_name
Barcelona           2788
Deportivo Alavés     974
Name: type_name, dtype: int64


In [61]:
print(df_match.groupby("team_name")["duration"].mean())

team_name
Barcelona           1.351428
Deportivo Alavés    1.109614
Name: duration, dtype: float64


In [62]:
print(df_match.groupby("team_name")["duration"].mean())          # agg-style
print()
print(df_match.groupby("team_name")["duration"].transform("mean"))  # transform-style

team_name
Barcelona           1.351428
Deportivo Alavés    1.109614
Name: duration, dtype: float64

0       1.351428
1       1.109614
2       1.351428
3       1.109614
4       1.109614
          ...   
3757    1.109614
3758    1.351428
3759    1.109614
3760    1.351428
3761    1.109614
Name: duration, Length: 3762, dtype: float64


In [63]:
df_match["team_avg_duration"] = df_match.groupby("team_name")["duration"].transform("mean")
df_match["duration_vs_team_avg"] = df_match["duration"] - df_match["team_avg_duration"]
print(df_match[["team_name", "duration", "team_avg_duration", "duration_vs_team_avg"]].head(10))

          team_name  duration  team_avg_duration  duration_vs_team_avg
0         Barcelona  0.000000           1.351428             -1.351428
1  Deportivo Alavés  0.000000           1.109614             -1.109614
2         Barcelona  0.000000           1.351428             -1.351428
3  Deportivo Alavés  0.000000           1.109614             -1.109614
4  Deportivo Alavés  2.015669           1.109614              0.906055
5  Deportivo Alavés       NaN           1.109614                   NaN
6  Deportivo Alavés  1.273280           1.109614              0.163666
7  Deportivo Alavés  3.287421           1.109614              2.177807
8  Deportivo Alavés       NaN           1.109614                   NaN
9  Deportivo Alavés  0.000000           1.109614             -1.109614


In [64]:
team_info = pd.DataFrame({
    "team_name": ["Barcelona", "Deportivo Alavés"],
    "stadium": ["Camp Nou", "Mendizorroza"],
    "founded": [1899, 1921]
})
print(team_info)
print()
df_merged = df_match.merge(team_info, on="team_name")
print(df_merged[["team_name", "type_name", "stadium", "founded"]].head())

          team_name       stadium  founded
0         Barcelona      Camp Nou     1899
1  Deportivo Alavés  Mendizorroza     1921

          team_name    type_name       stadium  founded
0         Barcelona  Starting XI      Camp Nou     1899
1  Deportivo Alavés  Starting XI  Mendizorroza     1921
2         Barcelona   Half Start      Camp Nou     1899
3  Deportivo Alavés   Half Start  Mendizorroza     1921
4  Deportivo Alavés         Pass  Mendizorroza     1921


In [65]:
grouped = df_match.groupby(["player_name", "type_name"])
print(grouped.size())

player_name                type_name     
Adrián Marín Gómez         Ball Receipt*      7
                           Ball Recovery      1
                           Carry              5
                           Foul Committed     2
                           Foul Won           1
                                             ..
Víctor Laguardia Cisneros  Dribbled Past      1
                           Duel               2
                           Interception       3
                           Pass              11
                           Pressure           4
Length: 285, dtype: int64


In [66]:
event_counts = df_match.groupby(["player_name", "type_name"]).size().reset_index(name="count")
print(event_counts.head(10))

                              player_name       type_name  count
0                      Adrián Marín Gómez   Ball Receipt*      7
1                      Adrián Marín Gómez   Ball Recovery      1
2                      Adrián Marín Gómez           Carry      5
3                      Adrián Marín Gómez  Foul Committed      2
4                      Adrián Marín Gómez        Foul Won      1
5                      Adrián Marín Gómez      Miscontrol      1
6                      Adrián Marín Gómez            Pass      7
7                      Adrián Marín Gómez        Pressure      4
8                      Adrián Marín Gómez            Shot      1
9  Arthur Henrique Ramos de Oliveira Melo   Ball Receipt*     17


In [67]:
event_wide = event_counts.pivot(index="player_name", columns="type_name", values="count")
print(event_wide.head())

type_name                               Bad Behaviour  Ball Receipt*  \
player_name                                                            
Adrián Marín Gómez                                NaN            7.0   
Arthur Henrique Ramos de Oliveira Melo            NaN           17.0   
Arturo Erasmo Vidal Pardo                         NaN            7.0   
Borja González Tomás                              NaN            4.0   
Daniel Alejandro Torres Rojas                     NaN           11.0   

type_name                               Ball Recovery  Block  Carry  \
player_name                                                           
Adrián Marín Gómez                                1.0    NaN    5.0   
Arthur Henrique Ramos de Oliveira Melo            NaN    NaN   15.0   
Arturo Erasmo Vidal Pardo                         NaN    NaN    6.0   
Borja González Tomás                              NaN    NaN    3.0   
Daniel Alejandro Torres Rojas                     1.0    2.0    9.0  

In [68]:
event_wide = event_wide.fillna(0)
print(event_wide.head())

type_name                               Bad Behaviour  Ball Receipt*  \
player_name                                                            
Adrián Marín Gómez                                0.0            7.0   
Arthur Henrique Ramos de Oliveira Melo            0.0           17.0   
Arturo Erasmo Vidal Pardo                         0.0            7.0   
Borja González Tomás                              0.0            4.0   
Daniel Alejandro Torres Rojas                     0.0           11.0   

type_name                               Ball Recovery  Block  Carry  \
player_name                                                           
Adrián Marín Gómez                                1.0    0.0    5.0   
Arthur Henrique Ramos de Oliveira Melo            0.0    0.0   15.0   
Arturo Erasmo Vidal Pardo                         0.0    0.0    6.0   
Borja González Tomás                              0.0    0.0    3.0   
Daniel Alejandro Torres Rojas                     1.0    2.0    9.0  